In [3]:
import numpy as np

In [4]:
def sparse_subspace_rotation(U, iterations=100, step_size=0.01):
    """
    Rotates basis U to find a sparse representation using L1 minimization
    and QR-based orthogonal projection.
    """
    n, k = U.shape
    # Initialize Q as an identity matrix (the starting rotation)
    Q = np.eye(k)
    
    for i in range(iterations):
        # 1. Compute current rotated basis: U_tilde = U @ Q
        U_tilde = U @ Q
        
        # 2. Gradient of the L1 norm: sign(U_tilde)
        # We want to move Q in a direction that reduces ||U @ Q||_1
        grad = U.T @ np.sign(U_tilde)
        
        # 3. Take a step to get the "draft" matrix Z (from the whiteboard)
        Z = Q - step_size * grad
        
        # 4. QR Decomposition to project Z back to an orthogonal matrix
        # This is the "Q, R = qr(Z)" step from the drawing
        Q_new, R = np.linalg.qr(Z)
        
        # Update Q
        Q = Q_new
        
        # Optional: Print progress (L1 norm should decrease)
        if i % 10 == 0:
            sparsity_score = np.sum(np.abs(U @ Q))
            print(f"Iteration {i}: L1 Norm = {sparsity_score:.4f}")
            
    return U @ Q, Q

# Example Usage:
# U = your_matrix_of_vectors (n x k)
# U_sparse, rotation_matrix = sparse_subspace_rotation(U)

In [5]:
U = np.eye(10)
# generate orthogonal matrix
Q = np.random.randn(10, 10)
Q = np.linalg.qr(Q)[0]
# rotate basis
U = U @ Q
U

array([[-0.00311833,  0.3317711 , -0.25430414, -0.01418821, -0.63114513,
         0.54628556,  0.1876498 , -0.27601184,  0.12572789, -0.03273741],
       [-0.4746574 , -0.32241161,  0.56564963, -0.41272951, -0.18691415,
         0.23576081,  0.01654518,  0.11347497,  0.23606829,  0.14507703],
       [-0.25671108,  0.22291041, -0.15861893, -0.30590895,  0.01561493,
        -0.57005515,  0.1397438 , -0.51943632,  0.3155537 ,  0.22703935],
       [ 0.33165246, -0.25415756,  0.02844029, -0.1615732 ,  0.24755158,
         0.25747285, -0.45685224, -0.49523391,  0.35981145, -0.29578185],
       [ 0.32361701,  0.08499299,  0.20105008, -0.61375672,  0.00115282,
        -0.02328774,  0.25987365, -0.16700992, -0.57859128, -0.2004785 ],
       [-0.48691357, -0.03738909,  0.19198338,  0.40808657,  0.16822912,
         0.1371426 ,  0.00370321, -0.54177096, -0.45356063, -0.10848551],
       [-0.21748486, -0.49935779, -0.67877451, -0.27697701,  0.12784323,
         0.19935203,  0.00568119,  0.01516707

In [6]:
# sparse subspace rotation
U_sparse, rotation_matrix = sparse_subspace_rotation(U, iterations=1000, step_size=0.01)
with np.printoptions(precision=4, suppress=True):
    print(U_sparse)


Iteration 0: L1 Norm = 25.7040
Iteration 10: L1 Norm = 24.5081
Iteration 20: L1 Norm = 23.6734
Iteration 30: L1 Norm = 23.0638
Iteration 40: L1 Norm = 22.1855
Iteration 50: L1 Norm = 20.9344
Iteration 60: L1 Norm = 19.0706
Iteration 70: L1 Norm = 17.2830
Iteration 80: L1 Norm = 16.2396
Iteration 90: L1 Norm = 15.3122
Iteration 100: L1 Norm = 14.5373
Iteration 110: L1 Norm = 13.6308
Iteration 120: L1 Norm = 13.0805
Iteration 130: L1 Norm = 12.8913
Iteration 140: L1 Norm = 12.5796
Iteration 150: L1 Norm = 12.2577
Iteration 160: L1 Norm = 11.8403
Iteration 170: L1 Norm = 11.5181
Iteration 180: L1 Norm = 11.2217
Iteration 190: L1 Norm = 11.1401
Iteration 200: L1 Norm = 11.1179
Iteration 210: L1 Norm = 11.0648
Iteration 220: L1 Norm = 11.0283
Iteration 230: L1 Norm = 10.9925
Iteration 240: L1 Norm = 10.9150
Iteration 250: L1 Norm = 10.8100
Iteration 260: L1 Norm = 10.6794
Iteration 270: L1 Norm = 10.5458
Iteration 280: L1 Norm = 10.3616
Iteration 290: L1 Norm = 10.3627
Iteration 300: L1 Nor

In [7]:
def varimax(U, iterations=1000, tol=1e-8):
    """
    Standard Varimax rotation: 
    Rotates U to maximize the variance of the squared loadings.
    """
    n, k = U.shape
    # Initialize Q as an identity matrix
    Q = np.eye(k)
    
    # Optional: Normalizing U (Kaiser Normalization)
    row_norms = np.sqrt(np.sum(U**2, axis=1, keepdims=True))
    U_norm = U / row_norms

    for i in range(iterations):
        Q_old = Q.copy()
        
        # 1. Rotate the basis: U_tilde = U @ Q
        U_tilde = U_norm @ Q
        
        # 2. Compute the Varimax objective gradient
        # (U_tilde**3) is the derivative of the L4 norm (sum of squared variances)
        A = U_norm.T @ (U_tilde**3 - (1.0/n) * U_tilde @ np.diag(np.sum(U_tilde**2, axis=0)))
        
        # 3. Use SVD to find the nearest orthogonal matrix (Procrustes Step)
        L, S, M_h = np.linalg.svd(A)
        Q = L @ M_h
        
        # Check convergence
        if np.abs(np.abs(np.diag(Q @ Q_old.T)).sum() - k) < tol:
            break

    # Final sparse representation
    U_sparse = (U_norm @ Q) * row_norms
    return U_sparse, Q

# Usage:
# U_sparse, Q_matrix = varimax(U)

In [8]:
U_sparse, Q_matrix = varimax(U)
# print L1 norm
display(np.sum(np.abs(U_sparse)))
with np.printoptions(precision=4, suppress=True):
    print(U_sparse)


np.float64(10.000031182037567)

[[ 0.  0.  0. -0. -1. -0.  0. -0.  0.  0.]
 [-1. -0. -0.  0. -0. -0.  0. -0. -0. -0.]
 [ 0. -0.  0. -0.  0. -1.  0.  0. -0. -0.]
 [ 0. -0.  0.  0.  0. -0. -0. -1. -0.  0.]
 [-0. -0.  0. -1.  0.  0. -0. -0. -0. -0.]
 [ 0.  0. -0.  0. -0.  0.  0.  0. -1.  0.]
 [ 0. -0. -1. -0. -0. -0.  0. -0.  0. -0.]
 [-0. -0. -0.  0. -0. -0. -1.  0. -0.  0.]
 [-0. -0. -0. -0.  0. -0.  0.  0.  0.  1.]
 [-0.  1. -0. -0.  0. -0. -0. -0.  0.  0.]]


In [15]:
def smoothed_sparse_rotation(U, iterations=1000, step_size=0.01, delta=0.01):
    n, k = U.shape
    Q = np.eye(k)
    
    for i in range(iterations):
        U_tilde = U @ Q
        
        # Smoothed L1 gradient (Huber-style)
        # Instead of np.sign, we use a gradient that scales down near zero
        grad = U.T @ (U_tilde / np.sqrt(U_tilde**2 + delta))
        
        # SVD Projection (the "Varimax Trick") for stability
        Z = Q - step_size * grad
        L, _, M_h = np.linalg.svd(Z)
        Q = L @ M_h
        
    return U @ Q, Q

In [13]:
U_sparse, Q_matrix = smoothed_sparse_rotation(U, iterations=1000)
print(np.sum(np.abs(U_sparse)))
with np.printoptions(precision=4, suppress=True):
    print(U_sparse)


10.000000000000025
[[-0. -0. -0. -0. -1.  0.  0.  0.  0. -0.]
 [-1. -0.  0. -0.  0. -0. -0. -0.  0.  0.]
 [-0. -0. -0.  0. -0. -1.  0. -0.  0. -0.]
 [ 0. -0. -0.  0. -0.  0. -0. -1.  0.  0.]
 [ 0. -0. -0. -1. -0. -0. -0. -0. -0.  0.]
 [-0. -0.  0.  0. -0. -0.  0. -0. -1. -0.]
 [-0.  0. -1.  0.  0.  0. -0.  0. -0. -0.]
 [ 0.  0. -0.  0. -0. -0. -1.  0. -0. -0.]
 [ 0.  0. -0.  0. -0. -0. -0.  0. -0.  1.]
 [-0.  1. -0. -0. -0.  0.  0.  0. -0. -0.]]
